We're continuing our Monte Carlo journey. Last lecture we focused on slab transmission of particles that were monodirectional and isotropic. Now let's start thinking of other physics we can add into the problem... 

<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> Let's say we want to add in scattering to this problem to calculate how many particles transit the shield without being absorbed. What things will we need to consider if we incorporate these physics into the problem? </h2>
</div>

Let's say the slab is made up of a material that has:
* $\Sigma_t$ = 2.0 cm$^{-1}$
* $\Sigma_s$ = 0.75 cm$^{-1}$
* $\Sigma_a$ = 1.25 cm$^{-1}$

Some things to consider:
* neutrons are scattered isotropically when they scatter (the direction can change to any other direction)
* a collision can either be a scatter OR an absorption, so we need to sample $\frac{\Sigma_s}{\Sigma_t}$ to see if it scatters.
  * if it does, then sample for another $\mu$ and keep following it. 

Our algorithm: 
* 
*
*
*
*
*

In [ ]:
import numpy as np

In [ ]:
def slab_transmission(Sig_s,Sig_a,thickness,N,isotropic=False):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
        Sig_s: The scattering macroscopic x-section
        Sig_a: The absorption macroscopic x-section
        thickness: Width of the slab
        N: Number of neutrons to simulate
        isotropic: Are the neutrons isotropic or a beam
    Returns:
        transmission: The fraction of neutrons that made it through
    """
    import numpy as np
    import math
    Sig_t = Sig_a + Sig_s
    iSig_t = 1/Sig_t
    transmission = 0.0
    N = int(N)
    for i in range(N):
        if (isotropic):
            mu = np.random.random()
        else:
            mu = 1.0
        x = 0
        alive = 1
        while (alive):
            #get distance to collision
            l = -math.log(1-np.random.random())*iSig_t
            #move particle
            x += l*mu
            #still in the slab?
            if (x>thickness):
                transmission += 1
                alive = 0
            elif (x<0):
                alive = 0
            else:
                #scatter or absorb
                if (np.random.random() < Sig_s*iSig_t):
                    #scatter, pick new mu
                    mu = np.random.uniform(-1,1)
                else: #absorbed
                    alive = 0
    transmission /= N
    return transmission

First, we can test this and see if we get roughly the same answer as our previous simulation. Use $\Sigma_s$=0.0 and $\Sigma_a$=2.0. 

In [ ]:
N = 1000000
Sigma_s = 0.0
Sigma_a = 2.0
thickness = 3.0
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, N,
                                 isotropic=True)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

Let's try another problem with scattering now.... 

In [ ]:
N = 1000000
Sigma_s = 0.75
Sigma_a = 2.0-Sigma_s
thickness = 3.0
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, N,
                                 isotropic=True)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> By adding scattering we see a different transmission. Did you expect this? How about the magnitude difference? </h2>
</div>

We can see as we stack more interactions into our problem, it can be more and more challenging to solve by hand. To solve this particular problem by hand is a bit out of scope of the course, but it would involve eigenfunction expansions. 

Let's move on to tracking particles in a different sort of geometry than a slab. 

**go to handwritten notes** 

With a spherical shell like this problem, we will have an interesting problem to deal with when a particle crosses back over the inner radius of the shell if it scatters. If it does, it will transit the hollow part of the sphere and then cross the inner radius again. We'll use ridder's method to solve this problem. 

In [ ]:
def ridder(f,a,b,epsilon=1.0e-6):
	"""Find the root of the function f via Ridder's Method where the root lies within [a,b]
	Args:
		f: function to find root of
		a: left-side of interval
		b: right-side of interval
		epsilon: tolerance
	Returns:
		estimate of root
	"""
	assert (b>a)
	assert (f(a)*f(b) < 0)
	delta = b - a
	iterations = 0
	residual = 1.0
	while (np.fabs(residual) > epsilon):
		c = 0.5*(b+a)
		d = 0.0
		if (f(a) - f(b) > 0):
			d = c + (c-a)*f(c)/np.sqrt(f(c)**2-f(a)*f(b))
		else:
			d = c - (c-a)*f(c)/np.sqrt(f(c)**2-f(a)*f(b))
		#now see which part of interval root is in
		if (f(a)*f(d) < 0):
			b=d
		elif (f(b)*f(d) < 0):
			a=d
		residual = f(d)
		iterations += 1
	#print("It took",iterations,"iterations")
	return d #return c

Ok, we have the method to solve the particle transit in the sphere. Let's define our problem. 

We will create neutrons at R$_i$. Let's set $z=R_i$ and $y=x=0$ initially. Because we have symmetry in our problem, we first limit our directional sampling to $\theta$ in $[0, \pi/2]$. 

In [ ]:
def shell_transmission(Sig_s,Sig_a,Ri,Ro,N):
    """Compute the fraction of neutrons that leak through a slab
        Inputs:
            Sig_s: The scattering macroscopic x-section
            Sig_a: The absorption macroscopic x-section
            Ri: Inner radius of the shell
            Ro: Outer radius of the shell
            N: Number of neutrons to simulate
        Returns:
            transmission: The fraction of neutrons that made it through
    """
    import numpy as np
    import math
    Sig_t = Sig_a + Sig_s
    iSig_t = 1/Sig_t
    transmission = 0.0
    N = int(N)
    for i in range(N):
        #get initial direction
        theta = np.random.uniform(0,0.5*np.pi)
        phi = np.random.uniform(0,2*np.pi)
        r = Ri
        z = Ri
        x = 0
        y = 0
        alive = 1
        #vector to keep track of positions
        xvec = x*np.ones([1])
        yvec = y*np.ones([1])
        zvec = z*np.ones([1])
        while (alive):
            #get distance to collision
            s = -math.log(1.0-np.random.random())*iSig_t
            #move particle
            z += s*math.cos(theta)
            y += s*math.sin(theta)*math.sin(phi)
            x += s*math.sin(theta)*math.cos(phi)
            xvec = np.append(xvec,x)
            yvec = np.append(yvec,y)
            zvec = np.append(zvec,z)
            r = math.sqrt(z**2 + y**2 + x**2)
            #still in the shell?
            if (r>Ro):
                transmission += 1
                alive = 0
            elif (r<Ri):
                #find s so that the neutron is on the other side of the shell
                f = lambda s: ((x + s*math.sin(theta)*math.cos(phi))**2 +
                               (y+s*math.sin(theta)*math.sin(phi))**2 +
                               (z + s*math.cos(theta))**2 - Ri**2)
                s = ridder(f,1e-10,2*Ri,1.0e-10)
                z += s*math.cos(theta)
                y += s*math.sin(theta)*math.sin(phi)
                x += s*math.sin(theta)*math.cos(phi)
                r = Ri
                #check that we are on the inner radius
                assert(math.fabs(x**2+y**2+z**2 - Ri**2) < 1e-6)
            else:
                #scatter or absorb
                if (np.random.random() < Sig_s*iSig_t):
                    #scatter, pick new angles
                    theta = np.random.uniform(0,math.pi)
                    phi = np.random.uniform(0,2*math.pi)
                else: #absorbed
                    alive = 0
    transmission /= N
    return transmission

Let's simulate this transmission through a shell of thickness 3cm with an inner radius of 2cm. Let's also choose to make scattering and absorption equally probable.

In [ ]:
N = 100
Sigma_s = 1.0
Sigma_a = 1.0
Ri = 2
Ro = Ri + 3
transmission = shell_transmission(Sigma_s,Sigma_a,Ri,Ro,N)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

Now let's test with more particles.... 

In [ ]:
N = 100000
Sigma_s = 1.0
Sigma_a = 1.0
Ri = 2
Ro = Ri + 3
transmission = shell_transmission(Sigma_s,Sigma_a,Ri,Ro,N)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

Ok, are we ready to make this even more fun???? 


Let's try to solve a realistic shielding problem. We'll use a $^{208}$Pb shield to shield a bare reactor made of $^{235}$U. We'll need to read in the data for the lead $\sigma_s$ and $\sigma_{capture}$, as well as get neutrons with a distribution of energies from fission (this is $\chi (E)$).

<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> What new physics are we adding to this problem? How will be have to change our sampling routines? </h2>
</div>



First, let's read in our cross sections in.... 

In [ ]:
import csv
lead_s = [] #create a blank list for the x-sects
lead_s_energy = [] #create a blank list for the x-sects energies
#this loop will only execute if the file opens
with open("pb_scat.csv") as csvfile:
    pbScat = csv.reader(csvfile)
    for row in pbScat: #have for loop that loops over each line
        lead_s.append(float(row[1]))
        lead_s_energy.append(float(row[0]))
    lead_scattering = np.array([lead_s_energy,lead_s])
    lead_abs = [] #create a blank list for the x-sects
    lead_abs_energy = [] #create a blank list for the x-sects energies
    #this loop will only execute if the file opens
with open("pb_radcap.csv") as csvfile:
    pbAbs = csv.reader(csvfile)
    for row in pbAbs: #have for loop that loops over each line
        lead_abs.append(float(row[1]))
        lead_abs_energy.append(float(row[0]))
        lead_absorption = np.array([lead_abs_energy,lead_abs])

We loaded the data in, but we'll want a lookup function to be able to query that cross section data based on the particles we're simulating. 

In [ ]:
def energy_lookup(data_set, inp_energy):
    """look up energy in a data set and return the nearest energy in the table
        Input:
            data_set: a vector of energies
            inp_energy: the energy to lookup
        Output:
            index: the index of the nearest neighbor in the table
    """
    #argmin returns the indices of the smallest members of an array
    #here we’ll look for the minimum difference
    #between the input energy and the table
    index = np.argmin(np.fabs(data_set-inp_energy))
    return index

The fission spectrum you may recall from when I wrote the transport equation in class last week. Typically our fission spectrum is written as:

$\chi (E) = 0.453 e^{-1.036 E} \sinh (\sqrt{2.29 E})$

let's write a function for that given some value of E that we pass to it. 

In [ ]:
def expfiss(x):
    import numpy as np
    return 0.453*np.exp(-1.036*x)*np.sinh(np.sqrt(2.29*x))

Ok, we have a lot of nice building blocks now. Let's talk a bit about rejection sampling and elastic scattering...... 



Now let's build up our slab reactor code... 

In [ ]:
def slab_reactor(sig_s,sig_a,thickness,density,A,N,isotropic=False):
    """Compute the fraction of neutrons that leak through a slab
        Inputs:
            sig_s: The scattering microscopic x-section array in form Energy,X-sect
            sig_a: The absorption microscopic x-section
            thickness: Width of the slab
            density: density of material in atoms per cc
            A: atomic weight of shield
            N: Number of neutrons to simulate
            isotropic: Are the neutrons isotropic or a beam
        Returns:
            transmission: energies of neutrons that leak through
            created: energies of neutrons that were born
    """
    import numpy as np
    import math
    import random
    alpha = (A-1.0)**2/(A+1.0)**2
    Sig_s = sig_s.copy()
    Sig_a = sig_a.copy()
    Sig_s[1,:] = density/1e24*Sig_s[1,:]
    Sig_a[1,:] = density/1e24*Sig_a[1,:]
    #make rejection box
    min_eng = np.min([np.min(Sig_s[0,:]),np.min(Sig_a[0,:])])
    max_eng = np.max([np.max(Sig_s[0,:]),np.max(Sig_a[0,:])])
    #max_prob = np.max([np.max(expfiss(Sig_a[0,:]/1000000.)),np.max(expfiss(Sig_s[0,:]/1000000.))])
    max_prob = 0.5
    transmission = []
    created = []
    N = int(N)
    for i in range(N):
        #sample direction
        if (isotropic):
            mu = np.random.random()
        else:
            mu = 1.0
        #compute energy via rejection sampling
        rejected = 1
        while (rejected):
            #pick x
            x = np.random.uniform(min_eng,max_eng)
            y = np.random.uniform(0,max_prob)
            rel_prob = expfiss(x/1000000.)
            if (y <= rel_prob):
                energy = x
                rejected = 0
        #initial position is 0
        x = 0
        created.append(energy)
        alive = 1
        while (alive):
            #get distance to collision
            scat_index = energy_lookup(Sig_s[0,:],energy)
            abs_index = energy_lookup(Sig_a[0,:],energy)
            cur_scat = Sig_s[1,scat_index]
            cur_abs = Sig_a[1,abs_index]
            Sig_t = cur_scat + cur_abs
            l = -math.log(1-np.random.random())/Sig_t
            #move particle
            x += l*mu
            #still in the slab
            if (x>thickness):
                transmission.append(energy)
                alive = 0
            elif (x<0):
                alive = 0
            else:
                #scatter or absorb
                if (random.random() < cur_scat/Sig_t):
                    #scatter, pick new mu and energy
                    mu = np.random.uniform(-1,1)
                    energy = np.random.uniform(alpha*energy,energy)
                else: #absorbed
                    alive = 0
    return transmission, created

In [ ]:
import math
N = 100
density = 11.34/208*6.022e23
thickness = 150
transmission,created = slab_reactor(lead_scattering,lead_absorption,thickness,density,208,N,isotropic=True)

In [ ]:
print(transmission)